# 階梯 1 教學模組：YouTube + YOLO + Colab — 辨識驗證 & 訓練串接

智慧製造 AI 實戰課程 · 進階視覺模組（補實作）。對應基礎架構 `ARCHITECTURE.md` 的階梯 1。

**雲端優先**：運算在 Colab T4，影片/資料/模型全放 Google Drive，本地零佔用。

- Part A 辨識驗證：公開 YouTube 影片 → COCO YOLO 偵測/追蹤 → 標註影片 + 偵測報告
- Part B 訓練串接：COCO 沒有的標的 → Roboflow 標註 → Colab T4 訓練 → 新影片驗證

執行前：Colab 選單 `執行階段 → 變更執行階段類型 → T4 GPU`。

## Part 0.0 · 先確認加速器（必跑第一格）

YOLO / PyTorch 用的是 **GPU (CUDA)**，不是 TPU。沒看到 GPU 就到 `執行階段 → 變更執行階段類型 → 硬體加速器 → T4 GPU` 再重跑。

In [ ]:
# 開跑前確認 GPU（Colab 已內建 torch；YOLO 走 CUDA，不用 TPU）
import torch, subprocess
print('PyTorch', torch.__version__, '| CUDA 可用:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('使用 GPU:', torch.cuda.get_device_name(0))
    subprocess.run(['nvidia-smi','--query-gpu=name,memory.total,memory.used','--format=csv'])
else:
    raise RuntimeError('沒有 GPU！執行階段 → 變更執行階段類型 → T4 GPU，再重跑。(TPU 對 YOLO 無效)')

## Part 0 · 環境（安裝 + 掛載 Drive）

In [ ]:
# 安裝（Colab 每次連線都要跑一次）
!pip -q install ultralytics yt-dlp
import ultralytics, torch
print('ultralytics', ultralytics.__version__, '| CUDA', torch.cuda.is_available())

In [ ]:
# 掛載 Google Drive，所有產物寫到 Drive（不佔 Colab 暫存）
from google.colab import drive
drive.mount('/content/drive')
import os
WORK = '/content/drive/MyDrive/yolo-course'   # 課程工作資料夾
for sub in ['videos','outputs','datasets','models']:
    os.makedirs(f'{WORK}/{sub}', exist_ok=True)
print('工作資料夾：', WORK)

## Part A · 辨識驗證（標準標的，免訓練）

標的物是 COCO 已知類別（人/車/巴士…），直接用預訓練模型驗證辨識流程能不能跑通。

In [ ]:
# === 參數（改這裡）===
YT_URL   = 'https://www.youtube.com/watch?v=MNn9qKG2UFI'  # 換成你的公開影片（含人/車的街景最佳）
MAX_SEC  = 20            # 只取前 N 秒，省 Drive 空間與時間
CONF     = 0.35          # 信心度門檻
CLASSES  = None          # None=全類別；聚焦可填 [0,2,5,7] = person,car,bus,truck
MODEL    = 'yolo11n.pt'  # COCO nano，最輕

In [ ]:
# 下載 YouTube 影片前 MAX_SEC 秒到 Drive
import subprocess, os
raw = f'{WORK}/videos/source.mp4'
clip = f'{WORK}/videos/clip.mp4'
!yt-dlp -f 'mp4[height<=720]' -o "$raw" "$YT_URL"
# 截前 MAX_SEC 秒（ffmpeg Colab 內建）
!ffmpeg -y -loglevel error -i "$raw" -t $MAX_SEC -c copy "$clip"
print('影片就緒：', clip, os.path.getsize(clip)//1024, 'KB')

In [ ]:
# 跑 YOLO 偵測 + ByteTrack 追蹤，輸出標註影片到 Drive
from ultralytics import YOLO
from collections import Counter
model = YOLO(MODEL)
counts = Counter()          # 各類別累積偵測數
track_ids = set()           # 不重複的追蹤 ID
results = model.track(clip, tracker='bytetrack.yaml', conf=CONF, classes=CLASSES,
                      stream=True, save=True, project=f'{WORK}/outputs', name='partA', exist_ok=True)
for r in results:
    for b in r.boxes:
        counts[model.names[int(b.cls)]] += 1
        if b.id is not None: track_ids.add(int(b.id))
print('完成。標註影片在', f'{WORK}/outputs/partA')

In [ ]:
# === 辨識驗證報告 ===
print('偵測類別累積次數（所有 frame 加總）:')
for name, n in counts.most_common():
    print(f'  {name:12} {n}')
print(f'\n不重複追蹤物件數（ByteTrack 唯一 ID）: {len(track_ids)}')
# 在 Colab 內嵌播放標註影片
import glob
from IPython.display import Video
out = sorted(glob.glob(f'{WORK}/outputs/partA/*.mp4') + glob.glob(f'{WORK}/outputs/partA/*.avi'))
print('輸出檔：', out)
Video(out[-1], embed=True, width=640) if out else print('找不到輸出影片')

### Part A 驗收（打通才算過）

- [ ] 標註影片裡目標物有被框出、框穩定跟著移動
- [ ] 偵測類別與影片內容相符（人/車有被認出）
- [ ] 信心度合理（多數 > 0.35）、不重複追蹤數約等於畫面中真實物件數

通過 = 「影片進 → 偵測出 → 驗證辨識」最小閉環打通。

## Part B · 訓練串接（專用標的，COCO 沒有的物件）

當你要偵測 COCO 沒有的標的（某零件、某產品），就走這條：
標註 → Colab T4 訓練 → 新影片驗證。這是階梯 3 工業流程的教學縮小版。

In [ ]:
# 取得資料集（擇一）
# 方式 1：Roboflow（資料不敏感最快）。在 roboflow.com 標 50-100 張匯出 YOLOv11 格式，貼下方程式
# !pip -q install roboflow
# from roboflow import Roboflow
# rf = Roboflow(api_key='YOUR_KEY')
# ds = rf.workspace('ws').project('proj').version(1).download('yolov11', location=f'{WORK}/datasets/custom')
# DATA = f'{WORK}/datasets/custom/data.yaml'
#
# 方式 2：自備資料集（手動上傳到 Drive datasets/custom/，內含 data.yaml + images/ + labels/）
DATA = f'{WORK}/datasets/custom/data.yaml'
print('資料集設定檔：', DATA)

In [ ]:
# Colab T4 訓練（沒有 DATA 資料集就跳過，不中斷 Run All）
import os
from ultralytics import YOLO
if not os.path.exists(DATA):
    print('跳過 Part B 訓練：找不到', DATA)
    print('要做訓練串接：先用 Roboflow 標 50-100 張匯出 YOLOv11，或上傳 data.yaml 到 datasets/custom/')
else:
    m = YOLO('yolo11n.pt')
    m.train(data=DATA, epochs=50, imgsz=640, batch=16,
            project=f'{WORK}/models', name='custom', exist_ok=True)
    print('best 權重：', f'{WORK}/models/custom/weights/best.pt')

In [ ]:
# 驗證（需先完成上一格訓練；沒 best.pt 就跳過）
import os
from ultralytics import YOLO
best_path = f'{WORK}/models/custom/weights/best.pt'
if not os.path.exists(best_path):
    print('跳過驗證：尚未訓練出 best.pt')
else:
    best = YOLO(best_path)
    metrics = best.val(data=DATA)
    print('mAP50:', round(metrics.box.map50, 3), '| mAP50-95:', round(metrics.box.map, 3))
    # 匯出邊緣格式給階梯3 RPi5： best.export(format='ncnn')

### Part B 驗收
- [ ] mAP50 > 0.7（教學門檻；資料品質越好越高）
- [ ] 在「沒看過的新影片」上仍能框出標的（泛化，非死記）

匯出邊緣格式（給階梯 3 RPi5）：`best.export(format='ncnn')`。

## 接續：居家 / 工業階梯

同一套 pipeline 換 profile 即切換階梯（見 repo `core/` + `profiles/`）：
```bash
python run.py --profile profiles/home_sensing.yaml    --source rtsp://...   # 居家
python run.py --profile profiles/factory_counting.yaml --source clip.mp4    # 工業
```
教學階梯把「影片→偵測→驗證→訓練」打通後，居家只是換 RTSP 來源 + 加規則，工業是換自訓練模型 + 計數 + 邊緣部署。